# 06 — Hybrid RAG

The **HybridStrategy** combines document retrieval *and* web search into a single pipeline. This is useful for complex questions that benefit from both curated knowledge and fresh information.

```
Query
  ├──→ VectorRetriever (top-3 docs)
  └──→ WebRetriever    (top-2 web results)
            │
            ▼
     Merge & Deduplicate
            │
            ▼
       Context → LLM → Answer
```

## 1. Imports

In [ ]:
import sys
sys.path.insert(0, "..")

try:
    from backend.adaptive_rag.strategies.hybrid_strategy import HybridStrategy
    from backend.adaptive_rag.retrievers.vector_retriever import VectorRetriever
    from backend.adaptive_rag.retrievers.web_retriever import WebRetriever
    print("✓ Imported HybridStrategy and retrievers")
except ImportError as e:
    print(f"⚠ Backend import: {e}")

## 2. Set Up Both Retrievers

In [ ]:
# Document store (same setup as Notebook 04)
from langchain.embeddings import FakeEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document

fake_emb = FakeEmbeddings(size=768)
docs = [
    Document(page_content="Transformers are a neural architecture introduced in 'Attention is All You Need' (2017).",
             metadata={"source": "deep_learning.txt"}),
    Document(page_content="GPT-4 is a large language model developed by OpenAI with multimodal capabilities.",
             metadata={"source": "llm.txt"}),
    Document(page_content="Retrieval-Augmented Generation (RAG) combines retrieval with generative models.",
             metadata={"source": "rag.txt"}),
]
vector_store = FAISS.from_documents(docs, fake_emb)
doc_retriever = VectorRetriever(vector_store)
print("✓ Document retriever ready")

# Web retriever (with fallback)
try:
    web_retriever = WebRetriever()
    print("✓ Web retriever ready (DuckDuckGo)")
except Exception:
    class MockWebRetriever:
        async def retrieve(self, query, top_k=2):
            return [{"content": f"Web result about {query}", "source": f"https://example.com", "relevance_score": 0.8}] * top_k
    web_retriever = MockWebRetriever()
    print("⚠ Using mock web retriever")

## 3. HybridStrategy in Action

In [ ]:
class MockLLM:
    async def generate_with_context(self, query, context):
        lines = context.split("\n")
        return f"Hybrid answer using {len(lines)} sources. \nKey points from: {lines[0][:60] if lines else 'N/A'}"

llm = MockLLM()
strategy = HybridStrategy(doc_retriever, web_retriever, llm)
print(f"✓ {strategy.name()}: {strategy.description()}")

## 4. Execute a Hybrid Query

In [ ]:
import asyncio

async def demo_hybrid():
    query = "How do Transformers work in modern LLMs?"
    result = await strategy.execute(query)
    
    print(f"Strategy: {result['strategy']}")
    print(f"Query   : {result['query']}")
    print(f"Answer  : {result['answer'][:300]}")
    print(f"\nAll sources ({len(result['sources'])}):")
    for s in result['sources']:
        print(f"  - {s['source'][:70]} (score: {s['score']:.2f})")

asyncio.run(demo_hybrid())

## 5. Source Merging Strategy

The hybrid strategy merges document results and web results by simple concatenation. In production, you'd add:

- **Deduplication** by source URL or content hash
- **Re‑ranking** with a cross‑encoder
- **Reciprocal Rank Fusion (RRF)** for balanced ranking

In [ ]:
async def inspect_merge():
    query = "AI safety regulations"
    result = await strategy.execute(query)
    
    total = result['documents']
    doc_count = sum(1 for d in total if "example.com" not in d.get("source", ""))
    web_count = sum(1 for d in total if "example.com" in d.get("source", ""))
    
    print(f"Document sources: {doc_count}")
    print(f"Web sources     : {web_count}")
    print(f"Total merged    : len(documents)")

asyncio.run(inspect_merge())

## 6. When to Use Hybrid

| Query Signal | Why Hybrid |
|-------------|-----------|
| `complexity=complex` + `is_time_sensitive=True` | Needs foundational docs *and* fresh data |
| `query_type=comparative` | Often benefits from both curated + current info |
| `needs_docs=True` + `needs_web=True` | Explicit signal from classifier |

> **Next:** [07 — Self-RAG Workflow](./07_Self_RAG_Workflow.ipynb) — iterative reflection & retrieval